# 02 - Location and Grades

We want to look at students' performance over the past 5 academic years, and see if there exists any relationships between a student's location and their grades.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
from src.mongo_connect import get_db

db = get_db()
print(db.name)

nexus


Many fields in the `students` collection are not final values, but rather “pointers” (ObjectIds) that refer to other collections.

If we use `df_students` directly for analysis, this information will be lost. So we first want to create a `df_student_full` where all pointers are resolved into actual values in advance, to help with later perform a "location vs. grades" analysis.

So we built a single `df_student_full`, filtered to the past 5 academic years (`startDate >= 2021-08-01`), where every field is an actual value instead of pointers.

Summary:
- `students.address` / `students.location` are not pointers. No join needed.
- `students.finalGrade` is a pointer (Object Id) into `finalgrades` -- one letter grade per student.
- `students.grades` is a list of pointers, but they resolve into `netmathgrades` (not the `grades` collection) -- each entry is one exam/assignment record with a `type` (`Exam`, `Final`, `Communication`, `Mastery`), a `score`, and a `status`.

## 1. Load students
(past 5 academic years, so `startDate >= 2021-08-01`)

In [2]:
Start_DATE = "2021-08-01"  # start of the 2021-2022 academic year

df_full = pd.DataFrame(list(db.students.find({
    "startDate": {"$gte": pd.Timestamp(Start_DATE)}
})))

print(df_full.shape)
df_full.head()

(6824, 54)


,_id,name,mathable,mentor,location,netId,email,otherEmails,defunctEmails,phones,...,completeDate,finalGrade,finalExam,hasCompletedQualtricsSurvey,courseOrientation,inactiveEmail,proctorUAccessCode,personal,accommodation,prairieLearn
0,60d1e14894fed001b0fc8a56,"{'first': 'Sharon', 'last': 'Wang'}",{'loginToken': ''},"{'name': {'first': 'Tayyab', 'last': 'Nawaz'},...","{'geocode': {'type': 'Point', 'coordinates': [...",sjwang3,sjwang3@illinois.edu,[wang.sharon93@gmail.com],[],[978-569-8189],...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,60d332cdcba1cd0254f595cb,"{'first': 'Kimaya', 'last': 'Urdhwareshe'}",{'loginToken': ''},"{'comments': [60e7314e99934403d80c69f8, 60d3d0...","{'geocode': {'type': 'Point', 'coordinates': [...",kimayau2,kimayau2@illinois.edu,[kimaya23@gmail.com],[],[669-234-6136],...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,60d5d5cb70f38d00d2e90fca,"{'first': 'Jian', 'last': 'Deng'}",{'loginToken': 'eyJhbGciOiJIUzI1NiIsInR5cCI6Ik...,"{'comments': [60e4d4bb99934403d80c68e4, 60dc74...","{'formatted': '1740 E 17th St, Brooklyn, NY 11...",jiand2,jiand2@illinois.edu,[jiandeng2519@gmail.com],[],[150-059-2007],...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,60e304c98e77b0046d3c8a83,"{'first': 'Xinyi', 'last': 'Jiang'}",{'loginToken': 'eyJhbGciOiJIUzI1NiIsInR5cCI6Ik...,"{'comments': [], 'name': {'first': 'Michael', ...","{'formatted': '22, Jalan Anggerik Vanda 31/168...",xinyij5,jxinyi18@gmail.com,[xinyij5@illinois.edu],[],[162-078-908],...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,60e5a7c9220e540547d752c2,"{'first': 'Merlin', 'last': 'Want'}",{'loginToken': ''},"{'name': {'first': 'Jason', 'last': 'Elliot'},...","{'geocode': {'type': 'Point', 'coordinates': [...",mwant2,merlinwant@gmail.com,"[mwant2@illinois.edu, thedweebconvention@gmail...",[],[202-258-1996],...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Flatten location fields
(already embedded on `students`, no join needed).

`address` -> `street` / `city` / `state` / `zip` / `country`.

Missing address fields are stored as empty strings `''` (not null), so we convert them to `None` -- otherwise `notna()` treats them as real values. Most international students have an empty `state`.

`state` is normalized to the 2-letter US code (e.g. `Illinois` -> `IL`).

Some `country` is blank for domestic students, so we fill it with "United States" only when `state` is a valid US code; otherwise we keep whatever country string is there, or `None` if there's no address at all.

`location.formatted` is kept as the geocoded full-address string.


In [3]:
def extract_address_field(addr, field):
    if isinstance(addr, dict):
        value = addr.get(field)
        if isinstance(value, str):
            return value.strip() or None  # treat '' as missing
        return value
    return None

#flatten the address dictionary into separate columns
df_full["street"] = df_full["address"].apply(lambda a: extract_address_field(a, "street"))
df_full["city"] = df_full["address"].apply(lambda a: extract_address_field(a, "city"))
df_full["state"] = df_full["address"].apply(lambda a: extract_address_field(a, "state"))
df_full["zip"] = df_full["address"].apply(lambda a: extract_address_field(a, "zip"))
df_full["country_raw"] = df_full["address"].apply(lambda a: extract_address_field(a, "country"))

df_full["formatted_location"] = df_full["location"].apply(
    lambda loc: loc.get("formatted") if isinstance(loc, dict) else None
)

# 50 states + DC, territories (PR, GU, VI, AS, MP) and military mail (AA, AE, AP)
US_STATE_CODES = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA", "HI", "ID", "IL", "IN", "IA",
    "KS", "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC", "SD", "TN", "TX", "UT", "VT",
    "VA", "WA", "WV", "WI", "WY", "DC",
    "PR", "GU", "VI", "AS", "MP", "AA", "AE", "AP",
}
STATE_NAME_TO_CODE = {"Illinois": "IL"}  # full names found in the data

df_full["state"] = df_full["state"].replace(STATE_NAME_TO_CODE)

def resolve_country(row):
    country = row["country_raw"]
    if isinstance(country, str):
        return country
    if row["state"] in US_STATE_CODES:
        return "United States" #fill "United States" only if state is a US state
    return None

df_full["country"] = df_full.apply(resolve_country, axis=1)
df_full = df_full.drop(columns=["country_raw"])

df_full[["street", "city", "state", "zip", "country", "formatted_location"]].head()


,street,city,state,zip,country,formatted_location
0,314 E White St Apt 304,Champaign,IL,61820-2249,United States,"314 E White St #304, Champaign, IL 61820, USA"
1,4341 Headen Way,Santa Clara,CA,95054-4134,United States,"4341 Headen Way, Santa Clara, CA 95054, USA"
2,1740 E 17th St,Brooklyn,NY,11229-2102,United States,"1740 E 17th St, Brooklyn, NY 11229, USA"
3,22 Jalan Anggerik Vanda 31 168,Shah Alam Selangor,None,40460,Malaysia,"22, Jalan Anggerik Vanda 31/168, Kota Kemuning..."
4,2814 Battery Pl NW,Washington,DC,20016-3439,United States,"2814 Battery Pl NW, Washington, DC 20016, USA"


## 3. Resolve `finalGrade` -> `finalgrades` (letter grade + GPA)
This involves a pointer, and needs to be solved:

The `finalGrade` in `Students` is a pointer (Object Id) into finalgrades -- one letter grade per student.

In [4]:
finalgrade_ids = df_full["finalGrade"].dropna().tolist()

df_fg = pd.DataFrame(list(db.finalgrades.find({"_id": {"$in": finalgrade_ids}})))[["_id", "grade"]]
df_fg = df_fg.rename(columns={"_id": "finalGrade", "grade": "finalGradeLetter"})

df_fg

,finalGrade,finalGradeLetter
0,611c08d92150aa001bd0320e,C+
1,613117e2130aee004a589498,A+
2,61311ec7130aee004a5894c0,A+
3,6144d0af7f219f001bcf29d6,A-
4,615c712437d91d001b11cacd,A+
...,...,...
4395,6a984b143b7fe9000780a6b1,A+
4396,6a9850c43b7fe9000780a7e6,A+
4397,6a98718c3b7fe9000780aaad,A
4398,6a989bc43b7fe9000780aba8,A+


In [5]:
df_full = df_full.merge(df_fg, on="finalGrade", how="left")

GRADE_TO_GPA = {
    "A+": 4.0, "A": 4.0, "A-": 3.7,
    "B+": 3.3, "B": 3.0, "B-": 2.7,
    "C+": 2.3, "C": 2.0, "C-": 1.7,
    "D+": 1.3, "D": 1.0, "D-": 0.7,
    "F": 0.0,
    # "W" (withdraw) and "Abs" -> NaN GPA
}
#creat a new column named "gpa" to store the matched GPA
df_full["gpa"] = df_full["finalGradeLetter"].map(GRADE_TO_GPA)

#Check how many students there are for each finalGradeLetter
df_full["finalGradeLetter"].value_counts(dropna=False)

finalGradeLetter
NaN    2424
A      1265
A+      955
A-      631
F       363
B       356
B+      333
B-      155
C       120
C+       94
C-       55
D        34
D+       22
D-       16
W         1
Name: count, dtype: int64

## 4. Resolve `grades[]` -> `netmathgrades`, pivot wide by exam type
4 `type` of exams occur:
- `Exam` (multiple exams through the course),
- `Final`,
- `Communication`,
- `Mastery`.

For each type we compute the mean `score` across a student's records of that type and the count of graded records (a `score` of null means submitted/scheduled but not graded yet).

In [6]:
student_ids = df_full["_id"].tolist()

df_ng = pd.DataFrame(list(db.netmathgrades.find(
    {"student": {"$in": student_ids}},
    {"student": 1, "type": 1, "score": 1, "status": 1}, #only care about these 4 in each record
)))
print("netmathgrades records pulled:", len(df_ng))
print(df_ng["type"].value_counts())

netmathgrades records pulled: 23212
type
Exam             14324
Final             6322
Communication     2411
Mastery            155
Name: count, dtype: int64


In [7]:
ng_agg = df_ng.groupby(["student", "type"])["score"].agg(["mean", "count"]).reset_index()
ng_wide = ng_agg.pivot(index="student", columns="type", values=["mean", "count"])
ng_wide.columns = [
    f"score_{typ}" if stat == "mean" else f"n_{typ}"
    for stat, typ in ng_wide.columns
]
ng_wide = ng_wide.reset_index().rename(columns={"student": "_id"})

df_full = df_full.merge(ng_wide, on="_id", how="left")


score_cols = [c for c in df_full.columns if c.startswith("score_") or c.startswith("n_")]
df_full[score_cols].describe()

,score_Communication,score_Exam,score_Final,score_Mastery,n_Communication,n_Exam,n_Final,n_Mastery
count,2385.000000,3523.000000,3724.000000,64.000000,2404.000000,6138.000000,6315.000000,155.000000
mean,90.674963,86.836885,87.531447,84.075469,0.995008,1.244379,0.590816,0.412903
std,8.180039,12.738022,14.335797,18.787168,0.103898,1.136262,0.493972,0.493952
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,87.000000,82.208333,82.500000,81.000000,1.000000,0.000000,0.000000,0.000000
50%,92.000000,90.333333,91.000000,89.750000,1.000000,2.000000,1.000000,0.000000
75%,96.100000,95.500000,96.000000,97.000000,1.000000,2.000000,1.000000,1.000000
max,100.000000,100.000000,200.000000,100.000000,2.000000,3.000000,2.000000,1.000000


## 5. Result: `df_student_full`
One row per student, filtered to the 5-year window, with location (`street`/`city`/`state`/`zip`/`country`), outcome (`finalGradeLetter`/`gpa`, `status`), and per-exam-type scores all resolved to real values -- no ObjectId pointers left.

In [8]:
df_student_full = df_full

key_cols = [
    "_id", "netId", "startDate", "status", "isPartner", "isAdult",
    "street", "city", "state", "zip", "country", "formatted_location",
    "finalGradeLetter", "gpa",
    "score_Exam", "n_Exam", "score_Final", "n_Final",
    "score_Communication", "n_Communication", "score_Mastery", "n_Mastery",
]

print("shape:", df_student_full.shape)
print("\nmissingness on the key columns:")
print(df_student_full[key_cols].isna().sum())

df_student_full[key_cols][
    df_student_full["finalGradeLetter"].notna() & df_student_full["score_Exam"].notna()
].head(10)

shape: (6824, 70)

missingness on the key columns:
_id                       0
netId                     0
startDate                 0
status                    0
isPartner                 0
isAdult                   0
street                    8
city                      8
state                  1038
zip                      43
country                  13
formatted_location        0
finalGradeLetter       2424
gpa                    2425
score_Exam             3301
n_Exam                  686
score_Final            3100
n_Final                 509
score_Communication    4439
n_Communication        4420
score_Mastery          6760
n_Mastery              6669
dtype: int64


,_id,netId,startDate,status,isPartner,isAdult,street,city,state,zip,...,finalGradeLetter,gpa,score_Exam,n_Exam,score_Final,n_Final,score_Communication,n_Communication,score_Mastery,n_Mastery
6,60f6cb4624990101964c34b3,tdb4,2021-08-02 05:00:00,Completed,False,True,36240 SW Viewridge Ln,Hillsboro,OR,97123-9006,...,C,2.0,71.666667,3.0,71.0,1.0,NaN,NaN,NaN,NaN
7,60f6cb4724990101964c34b5,rogers61,2021-08-02 05:00:00,Completed,False,True,545 35th St,Richmond,CA,94805-2123,...,A,4.0,84.000000,2.0,97.5,1.0,100.0,1.0,NaN,NaN
9,60f81cc58f183c01d941e4b6,pj17,2021-08-03 05:00:00,Completed,False,True,619 W 6th St,Wilton,IA,52778-9536,...,C,2.0,84.666667,3.0,61.5,1.0,NaN,NaN,NaN,NaN
10,60f81cc68f183c01d941e4b8,kjhaile2,2021-08-03 05:00:00,Completed,False,True,6051 W Touhy Ave,Chicago,IL,60646-1247,...,C+,2.3,72.666667,3.0,80.0,1.0,NaN,NaN,NaN,NaN
11,60f81cc68f183c01d941e4ba,jskent2,2021-08-03 05:00:00,Completed,False,True,4323 Fearrington Post,Pittsboro,NC,27312-5060,...,D,1.0,77.000000,3.0,52.0,1.0,NaN,NaN,NaN,NaN
12,60fac88819376f02a0f4e7cb,shijies3,2021-08-05 05:00:00,Completed,False,True,112 E John St Apt 205,Champaign,IL,61820,...,A-,3.7,94.666667,3.0,78.5,1.0,NaN,NaN,NaN,NaN
13,60fac88819376f02a0f4e7cd,tianjie3,2021-08-05 05:00:00,Completed,False,True,"Urbana, 1615 Melrose Park Ct Apt 2022",Urbana,IL,61801-0829,...,A+,4.0,98.666667,3.0,95.0,1.0,NaN,NaN,NaN,NaN
14,60fac88919376f02a0f4e7cf,bedford4,2021-08-05 05:00:00,Completed,False,True,15040 NW Oakhills Dr,Beaverton,OR,97006-5517,...,B-,2.7,85.500000,2.0,80.5,1.0,100.0,1.0,NaN,NaN
15,60fac88a19376f02a0f4e7d1,dmax2,2021-08-05 05:00:00,Completed,False,True,3506 N Hamilton Ave,Chicago,IL,60618-6121,...,A+,4.0,100.000000,3.0,96.0,1.0,NaN,NaN,NaN,NaN
18,6103fa4d168be400b20d5c8f,ajoerge2,2021-08-12 05:00:00,Completed,False,True,274 Georgetown Ave,Romeoville,IL,60446-4110,...,A+,4.0,100.000000,2.0,100.0,1.0,NaN,NaN,NaN,NaN


## 6. Location vs. finalgrade(gpa)
Average GPA by state / country / city, plus letter-grade distribution by location.

In [9]:
MIN_N = 15  # minimum students to trust a location's average GPA

# state-level analysis only makes sense for US students
df_analysis = df_student_full[
    df_student_full["gpa"].notna()
    & df_student_full["state"].notna()
    & (df_student_full["country"] == "United States")
].copy()

def gpa_by_group(df, group_col, min_n=MIN_N):
    g = (
        df.groupby(group_col)["gpa"]
        .agg(n="count", avg_gpa="mean", std_gpa="std")
        .sort_values("avg_gpa", ascending=False)
    )
    g["avg_gpa"] = g["avg_gpa"].round(3)
    g["std_gpa"] = g["std_gpa"].round(3)
    return g[g["n"] >= min_n]

gpa_by_state = gpa_by_group(df_analysis, "state")
print(f"states with n >= {MIN_N}: {len(gpa_by_state)} of {df_analysis['state'].nunique()} total")
gpa_by_state


states with n >= 15: 19 of 50 total


,n,avg_gpa,std_gpa
state,,,
IL,2719,3.540,0.790
WI,30,3.213,1.433
DC,34,3.100,1.394
CA,192,3.090,1.379
VA,106,3.015,1.414
MI,31,2.919,1.449
NJ,49,2.873,1.492
NY,90,2.832,1.471
MA,61,2.752,1.655


In [10]:
df_analysis_country = df_student_full[df_student_full["gpa"].notna() & df_student_full["country"].notna()].copy()

gpa_by_country = gpa_by_group(df_analysis_country, "country")
print(f"countries with n >= {MIN_N}: {len(gpa_by_country)} of {df_analysis_country['country'].nunique()} total")
gpa_by_country.head(35)

countries with n >= 15: 7 of 35 total


,n,avg_gpa,std_gpa
country,,,
Singapore,18,3.872,0.542
United States,3863,3.297,1.143
Canada,32,3.275,1.238
"China, Peoples Republic of",306,3.184,1.301
"Korea, South",38,3.047,1.487
India,31,2.942,1.586
United Kingdom,19,2.774,1.749


In [11]:
df_analysis_city = df_student_full[df_student_full["gpa"].notna() & df_student_full["city"].notna()].copy()

gpa_by_city = gpa_by_group(df_analysis_city, "city", min_n=5)
print(f"cities with n >= 5: {len(gpa_by_city)} of {df_analysis_city['city'].nunique()} total")
gpa_by_city.head(30)


cities with n >= 5: 113 of 874 total


,n,avg_gpa,std_gpa
city,,,
Jinan,6,4.000,0.000
Spring,5,4.000,0.000
Zhengzhou,5,4.000,0.000
Plainfield,14,3.957,0.109
Hangzhou,6,3.950,0.122
Berkeley,7,3.900,0.265
Germantown Hills,6,3.900,0.155
Fremont,5,3.880,0.164
Singapore,17,3.865,0.558


In [12]:
top10_states = df_analysis["state"].value_counts().head(10).index
grade_order = ["A+", "A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D+", "D", "D-", "F"]

dist = (
    df_analysis[df_analysis["state"].isin(top10_states)]
    .groupby(["state", "finalGradeLetter"])
    .size()
    .unstack(fill_value=0)
)
dist = dist.reindex(columns=[c for c in grade_order if c in dist.columns], fill_value=0)
dist_pct = dist.div(dist.sum(axis=1), axis=0).round(3) * 100
dist_pct.loc[top10_states]

finalGradeLetter,A+,A,A-,B+,B,B-,C+,C,C-,D+,D,D-,F
state,,,,,,,,,,,,,
IL,22.0,32.1,16.8,8.3,9.2,3.2,2.2,2.2,0.8,0.4,0.5,0.2,2.2
CA,26.0,24.0,12.0,6.8,6.8,1.6,2.1,2.6,3.6,0.5,0.5,0.5,13.0
VA,13.2,34.9,10.4,6.6,7.5,2.8,1.9,2.8,3.8,0.9,0.9,0.0,14.2
NY,22.2,18.9,12.2,3.3,8.9,3.3,3.3,5.6,1.1,3.3,1.1,1.1,15.6
TX,17.4,15.1,8.1,5.8,4.7,4.7,1.2,1.2,2.3,0.0,0.0,0.0,39.5
MA,23.0,26.2,8.2,4.9,3.3,3.3,0.0,3.3,1.6,0.0,1.6,3.3,21.3
NJ,24.5,24.5,4.1,4.1,4.1,12.2,4.1,2.0,0.0,0.0,4.1,0.0,16.3
PA,18.8,25.0,6.2,2.1,6.2,2.1,6.2,6.2,4.2,0.0,4.2,0.0,18.8
WA,15.9,15.9,13.6,9.1,9.1,4.5,2.3,2.3,4.5,0.0,2.3,0.0,20.5
